In [2]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import re

pd.set_option('display.max_columns', None)

In [11]:
# Load Data
dataset = load_dataset(
    "FronkonGames/steam-games-dataset",
    split='train'
)

df = dataset.to_pandas()

print(df.shape)
display(df.head())

(124146, 41)


,appID,name,release_date,estimated_owners,peak_ccu,required_age,price,dlc_count,detailed_description,short_description,supported_languages,full_audio_languages,reviews,header_image,website,support_url,support_email,windows,mac,linux,metacritic_score,metacritic_url,user_score,positive,negative,score_rank,achievements,recommendations,notes,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,developers,publishers,categories,genres,tags,screenshots,movies,packages
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,,,[],[],,https://shared.akamai.steamstatic.com/store_it...,,,,True,False,False,0,,0,0,0,,0,0,,0,0,0,0,[],[],[],[],[],[https://shared.akamai.steamstatic.com/store_i...,[],[]
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,0,"Springtime, April: when the cherry trees come ...","Spring has come, and our protagonist, Yukinari...",[English],[],,https://shared.akamai.steamstatic.com/store_it...,http://mangagamer.org/supipara,http://mangagamer.com,support@mangagamer.com,True,False,False,0,,0,252,3,,0,231,,8,0,8,0,[minori],[MangaGamer],"[Single-player, Steam Trading Cards, Steam Clo...",[Adventure],[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy Supipara - Chapter 1 Spring Ha..."
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,"Immerse yourself in the most beloved, mystical...",Discover an entrancing and spectacular world!,"[English, French, German, Russian]",[],,https://shared.akamai.steamstatic.com/store_it...,https://www.facebook.com/8FloorGames/,https://www.facebook.com/8FloorGames,support@8floor.net,True,True,False,0,,0,21,3,,0,0,,0,0,0,0,[Somer Games],[8floor],"[Single-player, Family Sharing]",[Casual],[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy Mystery Solitaire The Black Ra..."
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",Yuha! I'll start the broadcast! Hakko's extrem...,[Korean],[Korean],,https://shared.akamai.steamstatic.com/store_it...,,,yujingamesc@gmail.com,True,False,False,0,,0,0,0,,19,0,The game includes the following elements. 1. G...,0,0,0,0,[유진게임즈],[유진게임즈],"[Single-player, Steam Achievements, Family Sha...","[Casual, Indie, Simulation]",[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy \ubc84\ud29c\ubc84 \ud30c\ub77..."
4,3631080,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,[English],[English],,https://shared.akamai.steamstatic.com/store_it...,https://www.realityexpanded.com/books-games,https://www.realityexpanded.com,support@realityexpanded.com,True,False,False,0,,0,0,0,,0,0,,0,0,0,0,[Reality Expanded LLC],[Reality Expanded LLC],"[Single-player, VR Only, Steam Leaderboards, F...","[Action, Early Access]",[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy Maze Quest VR"", ""description"":..."


## Clean up columns

In [20]:
cols_to_drop = [
    'dlc_count',
    'header_image', 
    'website', 
    'support_url', 
    'support_email',
    'notes',
    'average_playtime_forever',
    'average_playtime_2weeks',
    'median_playtime_forever',
    'median_playtime_2weeks',
    'screenshots', 
    'movies', 
    'packages',
]

df1 = df.drop(columns=cols_to_drop)

df1['release_date'] = pd.to_datetime(df1['release_date'], errors='coerce')

df1['release_year'] = df1['release_date'].dt.year
df1['release_month'] = df1['release_date'].dt.month

df2 = df1[(df1['release_year'] >= 2015) & (df1['release_year'] < 2026)]
print(f"Rows remaning: {df2.shape[0]}")



Rows remaning: 119036


## Supported Languages

In [22]:
language_counts = Counter(
    lang
    for languages in df2["supported_languages"].dropna()
    for lang in languages
)

language_counts.most_common(20)

[('English', 105601),
 ('Simplified Chinese', 33302),
 ('Japanese', 28662),
 ('German', 28059),
 ('French', 27960),
 ('Russian', 26611),
 ('Spanish - Spain', 26195),
 ('Italian', 19251),
 ('Korean', 19180),
 ('Traditional Chinese', 17430),
 ('Portuguese - Brazil', 17393),
 ('Polish', 12833),
 ('Turkish', 10855),
 ('Portuguese - Portugal', 10831),
 ('Spanish - Latin America', 9657),
 ('Dutch', 7760),
 ('Ukrainian', 7173),
 ('Czech', 6378),
 ('Swedish', 6076),
 ('Thai', 5990)]

In [29]:
n = 100

top_languages = [
    lang for lang, count in language_counts.most_common(n)
]

top_languages

['English',
 'Simplified Chinese',
 'Japanese',
 'German',
 'French',
 'Russian',
 'Spanish - Spain',
 'Italian',
 'Korean',
 'Traditional Chinese',
 'Portuguese - Brazil',
 'Polish',
 'Turkish',
 'Portuguese - Portugal',
 'Spanish - Latin America',
 'Dutch',
 'Ukrainian',
 'Czech',
 'Swedish',
 'Thai',
 'Arabic',
 'Hungarian',
 'Danish',
 'Finnish',
 'Norwegian',
 'Vietnamese',
 'Romanian',
 'Greek',
 'Bulgarian',
 'Indonesian',
 'Hindi',
 'Catalan',
 'Malay',
 'Slovak',
 'Hebrew',
 'Serbian',
 'Croatian',
 'Filipino',
 'Lithuanian',
 'Persian',
 'Belarusian',
 'Latvian',
 'Basque',
 'Estonian',
 'Slovenian',
 'Icelandic',
 'Afrikaans',
 'Galician',
 'Bangla',
 'Georgian',
 'Kazakh',
 'Azerbaijani',
 'Welsh',
 'Albanian',
 'Irish',
 'Urdu',
 'Uzbek',
 'Swahili',
 'Bosnian',
 'Luxembourgish',
 'Macedonian',
 'Mongolian',
 'Gujarati',
 'Punjabi (Gurmukhi)',
 'Zulu',
 'Armenian',
 'Tamil',
 'Nepali',
 'Maori',
 'Igbo',
 'Xhosa',
 'Malayalam',
 'Telugu',
 'Kannada',
 'Marathi',
 'Hausa',


In [24]:
def clean_lang_name(lang):
    return re.sub(r"[^a-z0-9]+", "_", lang.lower()).strip("_")

for lang in top_languages:
    df2[f"lang_{clean_lang_name(lang)}"] = (
        df2["supported_languages"]
        .apply(lambda x: int(lang in x))
    )

top_languages_set = set(top_languages)

# other languages count
df2["other_langs"] = (
    df["supported_languages"]
    .apply(
        lambda langs: sum(
            lang not in top_languages_set
            for lang in langs
        )
    )
)

# total languages count
df2["num_supported_languages"] = (
    df2["supported_languages"]
    .apply(len)
)

lang_cols = [
    col for col in df2.columns
    if col.startswith("lang_")
]

df2[
    ["appID", "supported_languages"]
    + lang_cols
    + ["other_langs", "num_supported_languages"]
].head(10)

,appID,supported_languages,lang_english,lang_simplified_chinese,lang_japanese,lang_german,lang_french,lang_russian,lang_spanish_spain,lang_italian,lang_korean,lang_traditional_chinese,other_langs,num_supported_languages
0,2539430,[],0,0,0,0,0,0,0,0,0,0,0,0
1,496350,[English],1,0,0,0,0,0,0,0,0,0,0,1
2,1034400,"[English, French, German, Russian]",1,0,0,1,1,1,0,0,0,0,0,4
3,3292190,[Korean],0,0,0,0,0,0,0,0,1,0,0,1
4,3631080,[English],1,0,0,0,0,0,0,0,0,0,0,1
5,1654170,"[English, French, Italian, German, Spanish - S...",1,0,0,1,1,1,1,1,0,0,3,9
6,1934300,[English],1,0,0,0,0,0,0,0,0,0,0,1
7,700340,[English],1,0,0,0,0,0,0,0,0,0,0,1
8,1157670,"[English, French, Italian, German, Spanish - S...",1,1,1,1,1,1,1,1,1,1,19,29
9,1540330,"[English, French, Italian, German, Spanish - S...",1,1,1,1,1,1,1,1,1,1,20,30


## Publisher Encoding

In [25]:
# Create publisher_prior_games = (number of previously released games associated with the game's publisher(s))
df2 = df2.sort_values("release_date").reset_index(drop=True)

publisher_game_count = defaultdict(int)

df2["publisher_prior_games_max"] = 0
df2["publisher_prior_games_mean"] = 0.0
df2["publisher_first_release"] = 0

for date, group in df2.groupby("release_date", sort=True):

    # Calculate features using only games released BEFORE this date
    for idx in group.index:

        publishers = df2.at[idx, "publishers"]

        if len(publishers) > 0:
            counts = [
                publisher_game_count[p]
                for p in publishers
            ]

            df2.at[idx, "publisher_prior_games_max"] = max(counts)
            df2.at[idx, "publisher_prior_games_mean"] = np.mean(counts)
            df2.at[idx, "publisher_first_release"] = int(max(counts) == 0)

    # After features are calculated, add today's games
    # to publisher history
    for idx in group.index:

        publishers = df2.at[idx, "publishers"]

        if len(publishers) > 0:
            for publisher in publishers:
                publisher_game_count[publisher] += 1
    

In [26]:
# Check publisher results
df2[
    [
        "release_date",
        "publishers",
        "publisher_prior_games_max",
        "publisher_prior_games_mean",
        "publisher_first_release"
    ]
].sort_values(
    "publisher_prior_games_max",
    ascending=False
).head(20)

df2[
    [
        "publisher_prior_games_max",
        "publisher_prior_games_mean",
        "publisher_first_release"
    ]
].describe()

,publisher_prior_games_max,publisher_prior_games_mean,publisher_first_release
count,119036.000000,119036.000000,119036.000000
mean,8.057294,7.795999,0.505511
std,29.122682,28.673978,0.499972
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,1.000000
75%,3.000000,3.000000,1.000000
max,539.000000,539.000000,1.000000


In [27]:
df2.head()

,appID,name,release_date,estimated_owners,peak_ccu,required_age,price,detailed_description,short_description,supported_languages,full_audio_languages,reviews,windows,mac,linux,metacritic_score,metacritic_url,user_score,positive,negative,score_rank,achievements,recommendations,developers,publishers,categories,genres,tags,release_year,release_month,lang_english,lang_simplified_chinese,lang_japanese,lang_german,lang_french,lang_russian,lang_spanish_spain,lang_italian,lang_korean,lang_traditional_chinese,other_langs,num_supported_languages,publisher_prior_games_max,publisher_prior_games_mean,publisher_first_release
0,337070,"Dimmdrive :: Gaming Ramdrive @ 10,000+ MB/s",2015-01-01,50000 - 100000,5,0,14.99,Dimmdrive is a RAM-based hard drive software b...,"Play Steam Games @ 10,000+ MB/s :: Load games ...","[English, French, Italian, German, Spanish - S...",[],“Very functional and simple software designed ...,True,False,False,0,,0,280,168,,6,361,[Dimmdrive Inc.],[Dimmdrive Inc.],[Steam Achievements],[Utilities],[],2015,1,1,1,1,1,1,1,1,1,1,1,17,27,0,0.0,1
1,334300,Fake Colours,2015-01-01,100000 - 200000,0,0,1.49,"Designed around the RBG colour scheme, Fake Co...",The best new puzzle game around is ready to ch...,"[English, Korean, Portuguese - Brazil, Russian...",[],“FAKE COLOURS IS A GREAT PUZZLE GAME WITH A DI...,True,True,True,0,,0,131,26,,3,0,"[Beyond-Limits-Games, FCG]",[Budget Friendly Entertainment],"[Single-player, Steam Achievements, Steam Trad...","[Casual, Indie, Strategy]",[],2015,1,1,0,0,0,0,1,0,0,1,0,2,5,0,0.0,1
2,340570,WolfWars,2015-01-02,0 - 20000,0,0,2.99,WolfWars is a fantasy tactical strategy card g...,WolfWars is a fantasy tactical strategy card g...,[English],[English],,True,False,False,0,,0,9,7,,0,0,[Peter Hrdy],[Peter Hrdy],"[Single-player, Multi-player, Co-op, Cross-Pla...","[Casual, Indie, Simulation, Strategy, Early Ac...",[],2015,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0.0,1
3,338590,The Race for the White House,2015-01-02,0 - 20000,0,0,7.49,Play as a real-world candidate and completely ...,Play as a real-world candidate and completely ...,[English],[English],“It’s a laugh riot.” Washington Post,True,False,False,0,,0,30,19,,0,0,[Eversim],[Eversim],"[Single-player, Multi-player, Family Sharing]","[Indie, Simulation, Strategy]",[],2015,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0.0,1
4,340290,Dr.Green,2015-01-02,20000 - 50000,0,0,2.99,"Dr.Green is a 2D retro, arcade, action platfor...","Dr.Green is a 2D retro, arcade, action platfor...",[English],[],"“An arcade inspired action platformer, full of...",True,False,False,0,,0,27,14,,0,0,[Johan Aronsson],[Johan Aronsson],"[Single-player, Full controller support, Famil...","[Action, Indie]",[],2015,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0.0,1


In [28]:
# Game Age